# Phase 4: Controlled Evaluation

## Goal
Evaluate TF-IDF, LSA, and Word2Vec under the same split and report unified metrics.

## Required outputs
- unified result table
- classification metrics (macro-F1, accuracy)
- retrieval metrics (Recall@10, MRR@10)
- clustering metrics (NMI, ARI)
- saved tables in artifacts and reports folders

In [12]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity

# Resolve repository root whether kernel starts in repo root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src").exists() and (p / "configs").exists()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.metrics import classification_summary
from src.features.lsa_features import build_lsa
from src.features.tfidf_features import build_vectorizer
from src.models.linear_probe import build_logistic_probe
from src.models.train_word2vec import apply_row_cap, build_config, enforce_category_scope, load_split, _read_yaml
from src.utils.io import ensure_dir

PROJECT_ROOT

WindowsPath('C:/Coding/rep-learning-amazon-reviews')

In [13]:
params = _read_yaml(PROJECT_ROOT / "params.yaml")
exp_cfg = _read_yaml(PROJECT_ROOT / "configs" / "word2vec.yaml")
w2v_summary_path = PROJECT_ROOT / "artifacts" / "metrics" / "word2vec_skipgram_summary.json"
w2v_summary = json.loads(w2v_summary_path.read_text(encoding="utf-8"))

cli_stub = SimpleNamespace(
    max_train_rows=w2v_summary["config"].get("max_train_rows"),
    max_vector_rows=w2v_summary["config"].get("max_vector_rows"),
)
cfg = build_config(params=params, exp=exp_cfg, cli=cli_stub)

print("Phase 4 will align to Word2Vec run settings:")
print(json.dumps({
    "max_train_rows": cfg.max_train_rows,
    "max_vector_rows": cfg.max_vector_rows,
    "categories": cfg.categories,
    "strict_categories": cfg.strict_categories,
}, indent=2))

Phase 4 will align to Word2Vec run settings:
{
  "max_train_rows": null,
  "max_vector_rows": null,
  "categories": [
    "Electronics",
    "Home_and_Kitchen",
    "Beauty_and_Personal_Care",
    "Sports_and_Outdoors"
  ],
  "strict_categories": true
}


In [14]:
# Load text splits with the same logic as Phase 3 to keep representation comparison fair.
train_df, train_text_col = load_split(PROJECT_ROOT / "data" / "processed" / "train.parquet", cfg.text_column)
val_df, val_text_col = load_split(PROJECT_ROOT / "data" / "processed" / "val.parquet", cfg.text_column)
test_df, test_text_col = load_split(PROJECT_ROOT / "data" / "processed" / "test.parquet", cfg.text_column)

train_df = enforce_category_scope(train_df, cfg=cfg, split_name="train")
val_df = enforce_category_scope(val_df, cfg=cfg, split_name="val")
test_df = enforce_category_scope(test_df, cfg=cfg, split_name="test")

train_df = apply_row_cap(train_df, max_rows=cfg.max_train_rows, seed=cfg.seed)
val_df = apply_row_cap(val_df, max_rows=cfg.max_vector_rows, seed=cfg.seed)
test_df = apply_row_cap(test_df, max_rows=cfg.max_vector_rows, seed=cfg.seed)

print("Prepared split sizes:")
print(f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")
print("Text columns:", {"train": train_text_col, "val": val_text_col, "test": test_text_col})

Prepared split sizes:
train=1,395,911, val=198,948, test=405,141
Text columns: {'train': 'cleaned_text', 'val': 'cleaned_text', 'test': 'cleaned_text'}


In [15]:
# Load Word2Vec vectors and verify alignment with prepared labels.
X_train_w2v = np.load(PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_train_vectors.npy", mmap_mode="r")
X_val_w2v = np.load(PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_val_vectors.npy", mmap_mode="r")
X_test_w2v = np.load(PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_test_vectors.npy", mmap_mode="r")

assert X_train_w2v.shape[0] == len(train_df), "Train vector rows do not match train labels."
assert X_val_w2v.shape[0] == len(val_df), "Val vector rows do not match val labels."
assert X_test_w2v.shape[0] == len(test_df), "Test vector rows do not match test labels."

print("Word2Vec shapes:")
print("train", X_train_w2v.shape, "val", X_val_w2v.shape, "test", X_test_w2v.shape)

Word2Vec shapes:
train (1395911, 300) val (198948, 300) test (405141, 300)


In [16]:
# Build TF-IDF and LSA features on the same train/val/test rows used by Word2Vec.
tfidf_cap = 50000
tfidf_max_features = min(params.get("features", {}).get("tfidf_max_features", 100000), tfidf_cap)
lsa_components = params.get("features", {}).get("lsa_components", 300)

print(f"Using tfidf_max_features={tfidf_max_features}, lsa_components={lsa_components}")
tfidf = build_vectorizer(max_features=tfidf_max_features, ngram_range=(1, 2))
lsa = build_lsa(n_components=lsa_components)

X_train_tfidf = tfidf.fit_transform(train_df[train_text_col])
X_val_tfidf = tfidf.transform(val_df[val_text_col])
X_test_tfidf = tfidf.transform(test_df[test_text_col])

X_train_lsa = lsa.fit_transform(X_train_tfidf)
X_val_lsa = lsa.transform(X_val_tfidf)
X_test_lsa = lsa.transform(X_test_tfidf)

print("TF-IDF shapes:", X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape)
print("LSA shapes:", X_train_lsa.shape, X_val_lsa.shape, X_test_lsa.shape)

Using tfidf_max_features=50000, lsa_components=300
TF-IDF shapes: (1395911, 50000) (198948, 50000) (405141, 50000)
LSA shapes: (1395911, 300) (198948, 300) (405141, 300)


In [17]:
# Classification: sentiment and category on val/test for each representation.
y_train_sent = train_df["sentiment"].astype(str)
y_val_sent = val_df["sentiment"].astype(str)
y_test_sent = test_df["sentiment"].astype(str)

y_train_cat = train_df["main_category"].astype(str)
y_val_cat = val_df["main_category"].astype(str)
y_test_cat = test_df["main_category"].astype(str)

repr_data = {
    "tfidf": (X_train_tfidf, X_val_tfidf, X_test_tfidf),
    "lsa": (X_train_lsa, X_val_lsa, X_test_lsa),
    "word2vec": (X_train_w2v, X_val_w2v, X_test_w2v),
}

def fit_probe(X_train, y_train):
    clf = build_logistic_probe()
    clf.fit(X_train, y_train)
    return clf

def score_probe(clf, X_eval, y_eval):
    preds = clf.predict(X_eval)
    return classification_summary(y_eval, preds)

cls_rows = []
for rep_name, (Xtr, Xv, Xte) in repr_data.items():
    # Train once per task, then evaluate on both val and test splits.
    sent_clf = fit_probe(Xtr, y_train_sent)
    cat_clf = fit_probe(Xtr, y_train_cat)

    val_sent = score_probe(sent_clf, Xv, y_val_sent)
    test_sent = score_probe(sent_clf, Xte, y_test_sent)
    val_cat = score_probe(cat_clf, Xv, y_val_cat)
    test_cat = score_probe(cat_clf, Xte, y_test_cat)

    cls_rows.extend([
        {"representation": rep_name, "task": "sentiment", "split": "val", **val_sent},
        {"representation": rep_name, "task": "sentiment", "split": "test", **test_sent},
        {"representation": rep_name, "task": "category", "split": "val", **val_cat},
        {"representation": rep_name, "task": "category", "split": "test", **test_cat},
    ])

classification_df = pd.DataFrame(cls_rows).sort_values(["task", "split", "macro_f1"], ascending=[True, True, False])
classification_df

,representation,task,split,accuracy,macro_f1
3,tfidf,category,test,0.752072,0.751896
11,word2vec,category,test,0.666464,0.671784
7,lsa,category,test,0.620011,0.620679
2,tfidf,category,val,0.750121,0.750120
10,word2vec,category,val,0.666320,0.671801
6,lsa,category,val,0.620785,0.621471
1,tfidf,sentiment,test,0.891379,0.666380
9,word2vec,sentiment,test,0.857748,0.573703
5,lsa,sentiment,test,0.854658,0.569691
0,tfidf,sentiment,val,0.892268,0.665967


In [18]:
# Retrieval metrics using train set as retrieval database and val set as queries.
def retrieval_metrics(
    X_query,
    X_db,
    query_category,
    query_sentiment,
    db_category,
    db_sentiment,
    k=10,
    query_limit=1000,
    batch_size=128,
):
    n_queries = min(len(query_category), query_limit)
    recall_hits = 0
    reciprocal_sum = 0.0

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        sims = cosine_similarity(X_query[start:end], X_db)
        topk = np.argpartition(sims, -k, axis=1)[:, -k:]

        for i in range(end - start):
            q_idx = start + i
            candidates = topk[i]
            ranked = candidates[np.argsort(sims[i, candidates])[::-1]]

            positives = (
                (db_category.iloc[ranked].to_numpy() == query_category.iloc[q_idx])
                & (db_sentiment.iloc[ranked].to_numpy() == query_sentiment.iloc[q_idx])
            )

            if positives.any():
                recall_hits += 1
                first_rank = np.where(positives)[0][0] + 1
                reciprocal_sum += 1.0 / first_rank

    return {
        "recall_at_10": float(recall_hits / n_queries),
        "mrr_at_10": float(reciprocal_sum / n_queries),
        "queries": int(n_queries),
    }

retr_rows = []
for rep_name, (Xtr, Xv, _) in repr_data.items():
    metrics = retrieval_metrics(
        X_query=Xv,
        X_db=Xtr,
        query_category=y_val_cat,
        query_sentiment=y_val_sent,
        db_category=y_train_cat,
        db_sentiment=y_train_sent,
        k=10,
        query_limit=1000,
        batch_size=128,
    )
    retr_rows.append({"representation": rep_name, **metrics})

retrieval_df = pd.DataFrame(retr_rows).sort_values("mrr_at_10", ascending=False)
retrieval_df

,representation,recall_at_10,mrr_at_10,queries
2,word2vec,0.914,0.686144,1000
0,tfidf,0.876,0.546661,1000
1,lsa,0.880,0.530101,1000


In [19]:
# Clustering metrics on validation vectors.
def clustering_scores(X, category_labels, sentiment_labels, n_clusters=4, seed=42):
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, n_init="auto", batch_size=1024)
    pred = km.fit_predict(X)
    return {
        "nmi_category": float(normalized_mutual_info_score(category_labels, pred)),
        "ari_category": float(adjusted_rand_score(category_labels, pred)),
        "nmi_sentiment": float(normalized_mutual_info_score(sentiment_labels, pred)),
        "ari_sentiment": float(adjusted_rand_score(sentiment_labels, pred)),
    }

cluster_rows = []
for rep_name, (_, Xv, _) in repr_data.items():
    scores = clustering_scores(Xv, y_val_cat, y_val_sent, n_clusters=4, seed=cfg.seed)
    cluster_rows.append({"representation": rep_name, **scores})

clustering_df = pd.DataFrame(cluster_rows).sort_values("nmi_category", ascending=False)
clustering_df

,representation,nmi_category,ari_category,nmi_sentiment,ari_sentiment
0,tfidf,0.003325,0.000774,0.060444,-0.141579
2,word2vec,0.002327,0.001123,0.058874,-0.110413
1,lsa,0.001175,0.000410,0.017464,-0.002344


In [20]:
# Build a unified Phase 4 table and export all outputs.
test_cls = classification_df[classification_df["split"] == "test"]
sent_test = test_cls[test_cls["task"] == "sentiment"][["representation", "macro_f1", "accuracy"]].rename(columns={"macro_f1": "sentiment_test_macro_f1", "accuracy": "sentiment_test_accuracy"})
cat_test = test_cls[test_cls["task"] == "category"][["representation", "macro_f1", "accuracy"]].rename(columns={"macro_f1": "category_test_macro_f1", "accuracy": "category_test_accuracy"})

unified = sent_test.merge(cat_test, on="representation", how="inner")
unified = unified.merge(retrieval_df[["representation", "recall_at_10", "mrr_at_10"]], on="representation", how="left")
unified = unified.merge(clustering_df, on="representation", how="left")
unified = unified.sort_values("category_test_macro_f1", ascending=False)

metrics_dir = ensure_dir(PROJECT_ROOT / "artifacts" / "metrics")
tables_dir = ensure_dir(PROJECT_ROOT / "reports" / "tables")

classification_df.to_csv(metrics_dir / "phase4_classification_results.csv", index=False)
retrieval_df.to_csv(metrics_dir / "phase4_retrieval_results.csv", index=False)
clustering_df.to_csv(metrics_dir / "phase4_clustering_results.csv", index=False)
unified.to_csv(metrics_dir / "phase4_unified_results.csv", index=False)

classification_df.to_csv(tables_dir / "tbl_02_classification_results.csv", index=False)
retrieval_df.to_csv(tables_dir / "tbl_03_retrieval_results.csv", index=False)
clustering_df.to_csv(tables_dir / "tbl_04_clustering_results.csv", index=False)
unified.to_csv(tables_dir / "tbl_05_phase4_unified_results.csv", index=False)

print("Saved Phase 4 outputs to artifacts/metrics and reports/tables")
unified

Saved Phase 4 outputs to artifacts/metrics and reports/tables


,representation,sentiment_test_macro_f1,sentiment_test_accuracy,category_test_macro_f1,category_test_accuracy,recall_at_10,mrr_at_10,nmi_category,ari_category,nmi_sentiment,ari_sentiment
0,tfidf,0.666380,0.891379,0.751896,0.752072,0.876,0.546661,0.003325,0.000774,0.060444,-0.141579
1,word2vec,0.573703,0.857748,0.671784,0.666464,0.914,0.686144,0.002327,0.001123,0.058874,-0.110413
2,lsa,0.569691,0.854658,0.620679,0.620011,0.880,0.530101,0.001175,0.000410,0.017464,-0.002344


In [21]:
best_category = unified.sort_values("category_test_macro_f1", ascending=False).iloc[0]
best_sentiment = unified.sort_values("sentiment_test_macro_f1", ascending=False).iloc[0]
best_retrieval = unified.sort_values("mrr_at_10", ascending=False).iloc[0]
best_clustering = unified.sort_values("nmi_category", ascending=False).iloc[0]

print("Phase 4 interpretation")
print("- Best category classification:", best_category["representation"], f"({best_category['category_test_macro_f1']:.4f} macro-F1)")
print("- Best sentiment classification:", best_sentiment["representation"], f"({best_sentiment['sentiment_test_macro_f1']:.4f} macro-F1)")
print("- Best retrieval (MRR@10):", best_retrieval["representation"], f"({best_retrieval['mrr_at_10']:.4f})")
print("- Best clustering (NMI-category):", best_clustering["representation"], f"({best_clustering['nmi_category']:.4f})")

Phase 4 interpretation
- Best category classification: tfidf (0.7519 macro-F1)
- Best sentiment classification: tfidf (0.6664 macro-F1)
- Best retrieval (MRR@10): word2vec (0.6861)
- Best clustering (NMI-category): tfidf (0.0033)
